In [1]:
import os
import pandas as pd
import numpy as np
from dotenv import load_dotenv
from supabase import create_client, Client

# 1. 환경 변수 및 클라이언트 설정
load_dotenv()
url = os.environ.get("SUPABASE_URL")
key = os.environ.get("SUPABASE_KEY") # service_role 키 권장
supabase: Client = create_client(url, key)

def run_outlier_detection():
    # --- STEP 1: 분석 데이터 로드 ---
    # 실적, 활동량, 임계치 데이터를 모두 가져옵니다.
    usage_data = supabase.table("standard_usage").select("*").execute().data
    activity_data = supabase.table("activity_data").select("*").execute().data
    threshold_data = supabase.table("threshold_limits").select("*").execute().data
    
    usage_df = pd.DataFrame(usage_data)
    activity_df = pd.DataFrame(activity_data)
    threshold_df = pd.DataFrame(threshold_data)

    # 설정값 (노트북 참조)
    L1_Z_THRESHOLD, L1_YOY_THRESHOLD, L3_INTENSITY_THRESHOLD = 3.0, 30.0, 50.0

    print("🔍 이상치 탐지 시작...")

    for site in usage_df['site_id'].unique():
        for m_name in usage_df['metric_name'].unique():
            # 1. 특정 사업장/지표 데이터 필터링 및 정렬
            site_usage = usage_df[(usage_df['site_id'] == site) & (usage_df['metric_name'] == m_name)].sort_values('reporting_date')
            site_act = activity_df[activity_df['site_id'] == site].sort_values('reporting_date')
            
            # 실적과 활동량 병합
            merged = pd.merge(site_usage, site_act, on=['site_id', 'reporting_date'])
            
            # 해당 지표의 임계치(L2) 가져오기
            limit_row = threshold_df[(threshold_df['site_id'] == site) & (threshold_df['metric_name'] == m_name)]
            upper_limit = float(limit_row['upper_limit'].values[0]) if not limit_row.empty else float('inf')

            # 2. 검증 루프 (최근 12개월 데이터가 쌓인 시점부터 분석 가능)
            for i in range(12, len(merged)):
                row = merged.iloc[i]
                val, std_id = float(row['value']), int(row['id'])
                
                # [L1] 통계적 이상치 (Z-Score & YoY)
                window = merged.iloc[i-12:i]['value']
                z_score = abs(val - window.mean()) / window.std() if window.std() > 0 else 0
                yoy_val = merged.iloc[i-12]['value']
                yoy_roc = abs((val - yoy_val) / yoy_val * 100) if yoy_val != 0 else 0

                # [L2] 고정 임계치 초과 여부
                l2_fail = val > upper_limit

                # [L3] 원단위(Intensity) 이상치
                intensity = val / row['production_qty'] if row['production_qty'] > 0 else 0
                hist_int_mean = (merged.iloc[i-12:i]['value'] / merged.iloc[i-12:i]['production_qty']).mean()
                dev = abs((intensity - hist_int_mean) / hist_int_mean * 100) if hist_int_mean > 0 else 0

                # 3. 결과 판정 및 DB 업데이트
                layers = []
                if (z_score > L1_Z_THRESHOLD) or (yoy_roc > L1_YOY_THRESHOLD): layers.append(f"L1(Z:{z_score:.1f})")
                if l2_fail: layers.append(f"L2(Limit:{upper_limit})")
                if dev > L3_INTENSITY_THRESHOLD: layers.append(f"L3(Dev:{dev:.1f}%)")

                if layers:
                    # 이상치(2) 업데이트 및 결과 저장
                    severity = "Critical" if l2_fail else "Major" if dev > L3_INTENSITY_THRESHOLD else "Warning"
                    
                    # status 업데이트
                    supabase.table("standard_usage").update({"v_status": 2}).eq("id", std_id).execute()
                    
                    # 상세 사유 저장
                    outlier_log = {
                        "std_id": std_id,
                        "layer": ", ".join(layers),
                        "detected_value": float(val),
                        "threshold": float(upper_limit),
                        "severity": severity
                    }
                    supabase.table("outlier_results").insert(outlier_log).execute()
                else:
                    # 정상(1) 업데이트
                    supabase.table("standard_usage").update({"v_status": 1}).eq("id", std_id).execute()

    print("✅ 이상치 탐지 및 v_status 업데이트 완료!")

if __name__ == "__main__":
    run_outlier_detection()


🔍 이상치 탐지 시작...
✅ 이상치 탐지 및 v_status 업데이트 완료!
